In [ ]:
from __future__ import annotations

import pickle
from pathlib import Path

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.linalg import subspace_angles
from sklearn.decomposition import PCA

import manifold_dynamics.neural_utils as nu
import manifold_dynamics.paths as pth
import manifold_dynamics.tuning_utils as tut
import visionlab_utils.storage as vst

In [ ]:
TARGET = "19.Unknown.F"

# top_k value
topk_local = vst.fetch(f"{pth.OTHERS}/topk_vals.pkl")
with open(topk_local, "rb") as f:
    topk_vals = pickle.load(f)
top_k = int(topk_vals[TARGET]["k"])

# load in principal angle data
inpath = f"{pth.SAVEDIR}/dynamic_modes/shifting_subspace/{TARGET}.pkl"
local_inpath = vst.fetch(inpath)
with open(local_inpath, "rb") as f:
    principal_angles = pickle.load(f)
    
# actual raster data
raster_3d = np.nanmean(nu.significant_trial_raster(roi_uid=TARGET), axis=3)
image_order = np.asarray(tut.rank_images_by_response(raster_3d), dtype=int)
idx_topk = image_order[:top_k]

print(f"Successfully loaded data for {TARGET}...")

In [ ]:
early_window = slice(100, 150)
late_window  = slice(200, 300)

# slice time windows
early_resp = raster_3d[:, early_window, :1000]
late_resp  = raster_3d[:, late_window, :1000]

# average over units and time
early_mean = early_resp.mean(axis=(0, 1))  # shape: (images,)
late_mean  = late_resp.mean(axis=(0, 1))

early_top_idx = np.argsort(early_mean)[::-1][:top_k]
late_top_idx  = np.argsort(late_mean)[::-1][:top_k]


n = top_k
fig, axes = plt.subplots(2, n, figsize=(10, 2))
# Early
for i, idx in enumerate(early_top_idx[:n]):
    nu.plot_stimulus_image(idx, ax=axes[0, i])
    axes[0, i].axis('off')

# Late
for i, idx in enumerate(late_top_idx[:n]):
    nu.plot_stimulus_image(idx, ax=axes[1, i])
    axes[1, i].axis('off')

axes[0, 0].set_title("Early (50–125 ms)", loc='left')
axes[1, 0].set_title("Late (125–300 ms)", loc='left')

plt.tight_layout()
plt.show()

In [ ]:
pa = principal_angles['top']
processed_pa = np.nanmean(pa, axis=2)

sns.heatmap(processed_pa, cmap="viridis")

In [ ]:
natural_image_order = np.asarray(tut.rank_images_by_response(raster_3d[:, :, :1000]), dtype=int)
natural_idx_topk = natural_image_order[:top_k]
random_idx_topk = np.random.choice(natural_image_order[top_k:], top_k)

R_natural, _ = tut.tuning_rdm(raster_3d, natural_idx_topk)
R_random, _ = tut.tuning_rdm(raster_3d, random_idx_topk)
R_all, _ = tut.tuning_rdm(raster_3d, natural_image_order)

print(tut.ED2(R_natural), tut.ED2(R_random), tut.ED2(R_all))

fig, axes = plt.subplots(1,2, figsize=(6,2))
    
sns.heatmap(R_natural, square=True, vmax = 1, ax=axes[0])
sns.heatmap(R_all, square=True, vmax=1, ax=axes[1])